# CFPB Fraud Intelligence — Milestone 5 Final Evaluation (Colab)

This notebook runs the final reproducible experiment for the capstone:

1. clone the GitHub repository;
2. install the final dependencies;
3. verify the end-to-end pipeline;
4. compare a simple baseline prompt with the structured fraud-intelligence prompt;
5. calculate automatic grounding metrics;
6. create report-ready figures and a human-evaluation template.

**Recommended runtime:** T4 GPU.


In [ ]:
# Update these only if your repository name changes.
REPO_URL = "https://github.com/michaelbimo/Milestone-4-Model-Pipeline-Implementation.git"
BRANCH = "main"
PROJECT_FOLDER = "Milestone-4-Model-Pipeline-Implementation"
NUM_EVAL_SAMPLES = 30


## 1. Clone a clean copy of the repository

A clean clone makes the final evaluation reproducible and avoids depending on files left over from an earlier Colab session.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

project_dir = Path("/content") / PROJECT_FOLDER
if project_dir.exists():
    shutil.rmtree(project_dir)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)], check=True)
os.chdir(project_dir)
print("Working directory:", Path.cwd())


## 2. Install dependencies and confirm the runtime

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("The experiment can run on CPU, but a T4 GPU is recommended.")


## 3. Verify the required single-command pipeline

This is the same requirement demonstrated in Milestone 4. It should create the representative samples in `outputs/`.

In [ ]:
!python src/model_runner.py


In [ ]:
from pathlib import Path

for path in sorted(Path("outputs").glob("*")):
    if path.is_file():
        print(f"{path}: {path.stat().st_size:,} bytes")


## 4. Run the Milestone 5 experiment

The experiment uses the **same deterministic sample, model, seed, and decoding settings** for both conditions:

- `baseline`: simple complaint summarization prompt;
- `structured`: complaint + provisional archetype + red flags + risk level.

This makes the prompt comparison easier to interpret.

In [ ]:
!python src/experiment_runner.py --num-samples {NUM_EVAL_SAMPLES}


## 5. Evaluate the generated summaries

In [ ]:
!python src/evaluate.py


In [ ]:
import pandas as pd

comparison = pd.read_csv("results/model_comparison.csv")
display(comparison)


## 6. Display report-ready figures

In [ ]:
from IPython.display import Image, display

for figure in [
    "results/figures/prompt_comparison_metrics.png",
    "results/figures/summary_length_distribution.png",
]:
    print(figure)
    display(Image(filename=figure))


## 7. Human evaluation

Download `results/human_evaluation_template.csv` and rate a reasonable subset of outputs from **1–5** for factuality, relevance, clarity, and analyst usefulness.

After filling the ratings, replace the CSV in the repository/runtime and rerun:

```bash
python src/evaluate.py
```

The evaluator will automatically add the human-score averages to the final metrics when ratings are present.

## 8. Package the generated results for GitHub

Colab does **not** automatically push generated files back to GitHub. The safest workflow is to download the small final results ZIP, copy its contents into your local GitHub Desktop repository, then commit and push.

In [ ]:
import shutil
from pathlib import Path

archive_base = "/content/milestone5_results"
archive = shutil.make_archive(archive_base, "zip", root_dir=Path.cwd(), base_dir="results")
print("Download this file from the Colab Files panel:", archive)


### Recommended GitHub commit sequence

After copying the Milestone 5 code into the local repository, commit the code first:

```text
Add Milestone 5 evaluation pipeline
```

After the Colab experiment finishes and you copy the final `results/` files into the local repository, make a second commit:

```text
Add final prompt comparison results and figures
```

This gives the final repository a clear, professional history.